# Sistem Pendukung Keputusan Rekomendasi Laptop untuk Mahasiswa
**K-Means dan TOPSIS**

Alur: Preprocessing → CPU_Score & GPU_Score (benchmark) → StandardScaler → K-Means → TOPSIS → Ranking

| NIM | Nama |
|---|---|
| 412024034 | Jonathan Siswoyo |
| 412024040 | Zakharia Melkisedek Maranatha Nababan |
| 412025005 | Juan Raffles Fernando |
| 412025006 | Michael Valentino Susanto |

**Cara menjalankan:** buka di Google Colab dan jalankan sel dari atas ke bawah. Saat diminta, upload dataset laptop (bagian 1) dan dua file benchmark CPU/GPU (bagian 7). Daftar file ada di `data/raw/README.md`.

## 1. Load data
Upload file dataset laptop (`.csv` atau `.xlsx`) saat sel ini dijalankan.

In [1]:
import pandas as pd
import numpy as np
import re
from google.colab import files

uploaded = files.upload()  # pilih file dataset laptop
nama_file = list(uploaded)[0]

if nama_file.endswith('.xlsx'):
    df = pd.read_excel(nama_file)
else:
    df = pd.read_csv(nama_file, encoding='latin1')

df.columns = df.columns.str.strip()

print("Ukuran dataset:", df.shape)
print("Duplikat:", df.duplicated().sum())
print("\nMissing value:\n", df.isnull().sum())
df.head()

Saving laptops.csv.xlsx to laptops.csv.xlsx
Ukuran dataset: (1303, 12)
Duplikat: 28

Missing value:
 Manufacturer        0
Model Name          0
Category            0
Screen Size         0
Screen              0
CPU                 0
RAM                 0
Storage             0
GPU                 0
Operating System    0
Weight              0
Price (Euros)       0
dtype: int64


,Manufacturer,Model Name,Category,Screen Size,Screen,CPU,RAM,Storage,GPU,Operating System,Weight,Price (Euros)
0,Apple,MacBook Pro,Ultrabook,"13.3""",IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,"1339,69"
1,Apple,Macbook Air,Ultrabook,"13.3""",1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,"898,94"
2,HP,250 G6,Notebook,"15.6""",Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,"575,00"
3,Apple,MacBook Pro,Ultrabook,"15.4""",IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,"2537,45"
4,Apple,MacBook Pro,Ultrabook,"13.3""",IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,"1803,60"


## 2. Bersihkan data dasar
Hapus duplikat, hapus kolom yang tidak dipakai, ganti nama kolom.

In [3]:
df = df.drop_duplicates().drop(columns=['Operating System Version'], errors='ignore')

df = df.rename(columns={
    'Manufacturer': 'Brand',
    'Model Name': 'Model',
    'Screen Size': 'Screen_Size',
    'Operating System': 'OS',
    'Storage': 'Storage_Original',
    'RAM': 'RAM_GB',
    'Weight': 'Weight_KG',
    'Price (Euros)': 'Price_EUR',
})
df.columns.tolist()

['Brand',
 'Model',
 'Category',
 'Screen_Size',
 'Screen',
 'CPU',
 'RAM_GB',
 'Storage_Original',
 'GPU',
 'OS',
 'Weight_KG',
 'Price_EUR']

## 3. Ubah teks menjadi angka
Contoh: `8GB` → `8`, `1.37kg` → `1.37`, `1339,69` → `1339.69`.

In [4]:
# astype(str) supaya aman, baik kolomnya berupa teks (CSV) maupun angka (Excel)
df['RAM_GB'] = pd.to_numeric(df['RAM_GB'].astype(str).str.replace('GB', ''), errors='coerce')
df['Weight_KG'] = pd.to_numeric(df['Weight_KG'].astype(str).str.replace('kg', ''), errors='coerce')
df['Screen_Size'] = pd.to_numeric(df['Screen_Size'].astype(str).str.replace('"', ''), errors='coerce')
df['Price_EUR'] = pd.to_numeric(df['Price_EUR'].astype(str).str.replace(',', '.'), errors='coerce')

# Konversi Euro ke Rupiah (ubah kurs sesuai kebutuhan)
KURS_EURO = 19000
df['Price_IDR'] = df['Price_EUR'] * KURS_EURO

df[['RAM_GB', 'Weight_KG', 'Screen_Size', 'Price_EUR', 'Price_IDR']].head()

,RAM_GB,Weight_KG,Screen_Size,Price_EUR,Price_IDR
0,8,1.37,13.3,1339.69,25454110.0
1,8,1.34,13.3,898.94,17079860.0
2,8,1.86,15.6,575.00,10925000.0
3,16,1.83,15.4,2537.45,48211550.0
4,8,1.37,13.3,1803.60,34268400.0


## 4. Ekstrak fitur dari Storage, CPU, dan GPU

In [5]:
# Total kapasitas storage dalam GB (1 TB = 1024 GB)
def storage_gb(text):
    text = str(text).upper()
    tb = sum(float(x) for x in re.findall(r'([\d.]+)\s*TB', text))
    gb = sum(float(x) for x in re.findall(r'([\d.]+)\s*GB', text))
    return tb * 1024 + gb

# Cari kata kunci pertama yang cocok, kalau tidak ada -> 'Other'
def pick(text, rules):
    text = str(text).upper()
    for label, keywords in rules.items():
        if any(k in text for k in keywords):
            return label
    return 'Other'

storage_rules = {'SSD': ['SSD'], 'HDD': ['HDD'], 'Flash Storage': ['FLASH'], 'Hybrid': ['HYBRID']}
cpu_brand_rules = {'Intel': ['INTEL'], 'AMD': ['AMD']}
cpu_family_rules = {
    'Core i9': ['CORE I9'], 'Core i7': ['CORE I7'], 'Core i5': ['CORE I5'], 'Core i3': ['CORE I3'],
    'Ryzen 7': ['RYZEN 7'], 'Ryzen 5': ['RYZEN 5'], 'Ryzen 3': ['RYZEN 3'],
    'Celeron': ['CELERON'], 'Pentium': ['PENTIUM'], 'AMD A-Series': ['A-SERIES'],
}
gpu_brand_rules = {'NVIDIA': ['NVIDIA'], 'AMD': ['AMD', 'RADEON'], 'Intel': ['INTEL']}

df['Storage_GB'] = df['Storage_Original'].apply(storage_gb)
df['Storage_Type'] = df['Storage_Original'].apply(pick, rules=storage_rules)
df['CPU_Brand'] = df['CPU'].apply(pick, rules=cpu_brand_rules)
df['CPU_Family'] = df['CPU'].apply(pick, rules=cpu_family_rules)
df['GPU_Brand'] = df['GPU'].apply(pick, rules=gpu_brand_rules)

df[['Storage_Original', 'Storage_GB', 'Storage_Type', 'CPU', 'CPU_Family', 'GPU', 'GPU_Brand']].head(10)

,Storage_Original,Storage_GB,Storage_Type,CPU,CPU_Family,GPU,GPU_Brand
0,128GB SSD,128.0,SSD,Intel Core i5 2.3GHz,Core i5,Intel Iris Plus Graphics 640,Intel
1,128GB Flash Storage,128.0,Flash Storage,Intel Core i5 1.8GHz,Core i5,Intel HD Graphics 6000,Intel
2,256GB SSD,256.0,SSD,Intel Core i5 7200U 2.5GHz,Core i5,Intel HD Graphics 620,Intel
3,512GB SSD,512.0,SSD,Intel Core i7 2.7GHz,Core i7,AMD Radeon Pro 455,AMD
4,256GB SSD,256.0,SSD,Intel Core i5 3.1GHz,Core i5,Intel Iris Plus Graphics 650,Intel
5,500GB HDD,500.0,HDD,AMD A9-Series 9420 3GHz,Other,AMD Radeon R5,AMD
6,256GB Flash Storage,256.0,Flash Storage,Intel Core i7 2.2GHz,Core i7,Intel Iris Pro Graphics,Intel
7,256GB Flash Storage,256.0,Flash Storage,Intel Core i5 1.8GHz,Core i5,Intel HD Graphics 6000,Intel
8,512GB SSD,512.0,SSD,Intel Core i7 8550U 1.8GHz,Core i7,Nvidia GeForce MX150,NVIDIA
9,256GB SSD,256.0,SSD,Intel Core i5 8250U 1.6GHz,Core i5,Intel UHD Graphics 620,Intel


## 5. Tangani missing value
Baris yang beratnya kosong dihapus.

In [6]:
print("Missing value sebelum:\n", df.isnull().sum())

df = df.dropna(subset=['Weight_KG']).reset_index(drop=True)

print("\nMissing value sesudah:", df.isnull().sum().sum())
print("Ukuran akhir:", df.shape)

df.to_csv('laptops_cleaned.csv', index=False)

Missing value sebelum:
 Brand               0
Model               0
Category            0
Screen_Size         0
Screen              0
CPU                 0
RAM_GB              0
Storage_Original    0
GPU                 0
OS                  0
Weight_KG           1
Price_EUR           0
Price_IDR           0
Storage_GB          0
Storage_Type        0
CPU_Brand           0
CPU_Family          0
GPU_Brand           0
dtype: int64

Missing value sesudah: 0
Ukuran akhir: (1274, 18)


## 6. Buat daftar CPU & GPU untuk dicari skor benchmark-nya
Sel ini membuat dua file berisi daftar CPU/GPU unik dengan kolom skor yang masih kosong. Isi skornya (misalnya dari PassMark), lalu upload di langkah berikutnya.

*Kalau file benchmark yang sudah terisi sudah ada, lewati sel ini.*

In [7]:
def clean_text(s):
    return re.sub(r'\s+', ' ', str(s).strip())

cpu_list = pd.DataFrame({'CPU': sorted({clean_text(c) for c in df['CPU']})})
cpu_list['CPU_Score'] = np.nan

gpu_list = pd.DataFrame({'GPU_Search': sorted({clean_text(g) for g in df['GPU']})})
gpu_list['GPU_Score'] = np.nan

print("CPU unik:", len(cpu_list), "| GPU unik:", len(gpu_list))

cpu_list.to_excel('daftar_cpu_benchmark.xlsx', index=False)
gpu_list.to_excel('daftar_gpu_benchmark.xlsx', index=False)
files.download('daftar_cpu_benchmark.xlsx')
files.download('daftar_gpu_benchmark.xlsx')

CPU unik: 118 | GPU unik: 106


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Gabungkan skor benchmark ke dataset laptop
Upload dua file benchmark yang sudah terisi:
- `daftar_cpu_benchmark_FINAL.xlsx` (kolom `CPU`, `CPU_Score`)
- `daftar_gpu_benchmark_FINAL_BERES.xlsx` (kolom `GPU_Search`, `GPU_Score`)

Nama CPU/GPU dicocokkan setelah diseragamkan (huruf kecil, tanda `-` dan `_` jadi spasi).

In [8]:
files.upload()  # pilih kedua file benchmark

cpu = pd.read_excel("daftar_cpu_benchmark_FINAL.xlsx")
gpu = pd.read_excel("daftar_gpu_benchmark_FINAL_BERES.xlsx")

# Skor GPU yang diisi manual (tidak ditemukan di sumber benchmark)
manual_gpu = {
    "Intel HD Graphics 400": 300,
    "Intel HD Graphics 405": 350,
    "ARM Mali T860 MP4": 550,
}
for name, score in manual_gpu.items():
    gpu.loc[gpu["GPU_Search"] == name, "GPU_Score"] = score

# Seragamkan nama supaya bisa dicocokkan
def normalize_name(x):
    return " ".join(str(x).lower().replace("-", " ").replace("_", " ").split())

laptop = df.copy()
laptop["CPU_key"] = laptop["CPU"].apply(normalize_name)
laptop["GPU_key"] = laptop["GPU"].apply(normalize_name)
cpu["CPU_key"] = cpu["CPU"].apply(normalize_name)
gpu["GPU_key"] = gpu["GPU_Search"].apply(normalize_name)

# Gabungkan (left join: semua laptop tetap ada)
laptop = laptop.merge(cpu[["CPU_key", "CPU_Score"]].drop_duplicates("CPU_key"), on="CPU_key", how="left")
laptop = laptop.merge(gpu[["GPU_key", "GPU_Score"]].drop_duplicates("GPU_key"), on="GPU_key", how="left")

print("Jumlah laptop      :", len(laptop))
print("CPU tidak match    :", laptop["CPU_Score"].isna().sum())
print("GPU tidak match    :", laptop["GPU_Score"].isna().sum())

# Tampilkan CPU/GPU yang belum punya skor (kalau ada)
print(laptop.loc[laptop["CPU_Score"].isna(), "CPU"].drop_duplicates().to_string(index=False))
print(laptop.loc[laptop["GPU_Score"].isna(), "GPU"].drop_duplicates().to_string(index=False))

Saving daftar_cpu_benchmark_FINAL.xlsx to daftar_cpu_benchmark_FINAL.xlsx
Saving daftar_gpu_benchmark_FINAL_BERES.xlsx to daftar_gpu_benchmark_FINAL_BERES.xlsx
Jumlah laptop      : 1274
CPU tidak match    : 0
GPU tidak match    : 0
Series([], )
Series([], )


## 8. K-Means: kelompokkan laptop
Laptop dikelompokkan jadi 3 cluster berdasarkan CPU, GPU, RAM, dan storage. Data di-*scale* dulu supaya semua fitur punya bobot yang adil. Cluster diberi label **Rendah / Menengah / Tinggi** berdasarkan rata-rata spesifikasinya.

In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

features = ["CPU_Score", "GPU_Score", "RAM_GB", "Storage_GB"]

X_scaled = StandardScaler().fit_transform(laptop[features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
laptop["Cluster"] = kmeans.fit_predict(X_scaled)

# Urutkan cluster dari rata-rata spesifikasi terendah ke tertinggi
cluster_mean = pd.Series(X_scaled.mean(axis=1), index=laptop.index).groupby(laptop["Cluster"]).mean()
order = cluster_mean.sort_values().index
laptop["Cluster_Label"] = laptop["Cluster"].map(dict(zip(order, ["Rendah", "Menengah", "Tinggi"])))

print(laptop["Cluster_Label"].value_counts())
laptop.groupby("Cluster_Label")[features].mean().round(1)

Cluster_Label
Rendah      718
Menengah    421
Tinggi      135
Name: count, dtype: int64


,CPU_Score,GPU_Score,RAM_GB,Storage_GB
Cluster_Label,,,,
Menengah,5203.9,2205.5,10.0,783.1
Rendah,2676.3,812.7,5.9,406.6
Tinggi,7025.6,9872.9,17.2,1242.1


### Evaluasi K-Means
Clustering tidak punya label asli, jadi kualitasnya diukur dengan metrik internal:
- **Silhouette Score**: semakin mendekati 1, cluster semakin terpisah jelas.
- **Davies-Bouldin Index**: semakin kecil, semakin baik.

Label Rendah/Menengah/Tinggi adalah hasil pengelompokan, bukan label aktual dari dataset.

In [ ]:
# Evaluasi hasil K-Means (clustering tidak berlabel, jadi memakai metrik internal)
from sklearn.metrics import silhouette_score, davies_bouldin_score

print("Silhouette Score    :", round(silhouette_score(X_scaled, laptop["Cluster"]), 4))
print("Davies-Bouldin Index:", round(davies_bouldin_score(X_scaled, laptop["Cluster"]), 4))

## 9. TOPSIS: ranking laptop
Langkah-langkahnya:
1. Tentukan kriteria, jenis (*benefit* = makin besar makin baik, *cost* = makin kecil makin baik), dan bobot
2. Normalisasi matriks
3. Kalikan dengan bobot
4. Cari solusi ideal positif (terbaik) dan negatif (terburuk)
5. Hitung jarak tiap laptop ke kedua solusi ideal
6. Nilai preferensi = D⁻ / (D⁺ + D⁻). Makin besar, makin baik

In [10]:
# Kriteria: (jenis, bobot)
criteria = {
    "Price_IDR":  ("cost",    0.30),
    "CPU_Score":  ("benefit", 0.25),
    "GPU_Score":  ("benefit", 0.20),
    "RAM_GB":     ("benefit", 0.15),
    "Storage_GB": ("benefit", 0.10),
}
cols = list(criteria)
is_benefit = pd.Series({c: t == "benefit" for c, (t, w) in criteria.items()})
weights = pd.Series({c: w for c, (t, w) in criteria.items()})
print("Total bobot:", weights.sum())

# 1-2. Matriks keputusan -> normalisasi
X = laptop[cols]
R = X / np.sqrt((X ** 2).sum())

# 3. Normalisasi terbobot
V = R * weights

# 4. Solusi ideal positif & negatif
ideal_pos = V.max().where(is_benefit, V.min())
ideal_neg = V.min().where(is_benefit, V.max())

# 5. Jarak ke solusi ideal
d_pos = np.sqrt(((V - ideal_pos) ** 2).sum(axis=1))
d_neg = np.sqrt(((V - ideal_neg) ** 2).sum(axis=1))

# 6. Nilai preferensi
laptop["Preference_Value"] = d_neg / (d_pos + d_neg)

Total bobot: 1.0


## 10. Hasil ranking

In [11]:
ranking = laptop.sort_values("Preference_Value", ascending=False).reset_index(drop=True)
ranking["Ranking"] = ranking.index + 1

show = ["Ranking", "Brand", "Model", "Price_IDR", "CPU_Score", "GPU_Score",
        "RAM_GB", "Storage_GB", "Cluster_Label", "Preference_Value"]

print("10 laptop terbaik:")
ranking[show].head(10)

10 laptop terbaik:


,Ranking,Brand,Model,Price_IDR,CPU_Score,GPU_Score,RAM_GB,Storage_GB,Cluster_Label,Preference_Value
0,1,Asus,Rog Strix,29431000.0,14737,8782.0,16,1280.0,Tinggi,0.612532
1,2,Asus,G701VO-IH74K (i7-6820HK/32GB/2x,24301000.0,7005,7398.0,32,512.0,Tinggi,0.598659
2,3,Asus,Rog Strix,24681000.0,14737,8782.0,8,1280.0,Tinggi,0.595064
3,4,Asus,Rog Strix,41781000.0,14737,8782.0,16,1280.0,Tinggi,0.578170
4,5,Lenovo,IdeaPad Y910-17ISK,50597000.0,7005,13510.0,32,1280.0,Tinggi,0.577039
5,6,Asus,ROG G701VI,53181000.0,7005,15607.0,32,512.0,Tinggi,0.575400
6,7,MSI,GT80S 6QF-074US,53181000.0,7204,12523.0,32,1536.0,Tinggi,0.563984
7,8,Asus,FX503VM-E4007T (i7-7700HQ/16GB/1TB,27531000.0,6887,10021.0,16,1152.0,Tinggi,0.563385
8,9,Dell,Alienware 17,53200000.0,6527,13510.0,32,1280.0,Tinggi,0.562326
9,10,Dell,Inspiron 7577,28481000.0,6887,10021.0,16,1280.0,Tinggi,0.561792


### Komposisi cluster pada 10, 20, dan 50 besar
Melihat cluster apa yang mendominasi ranking teratas, dan laptop terbaik di tiap cluster.

In [12]:
# Komposisi cluster pada 10, 20, dan 50 besar
for n in [10, 20, 50]:
    print(f"\n{n} besar:")
    print(ranking.head(n)["Cluster_Label"].value_counts())

# Laptop terbaik di tiap cluster
print("\nTerbaik per cluster:")
ranking.groupby("Cluster_Label").head(1)[show]


10 besar:
Cluster_Label
Tinggi    10
Name: count, dtype: int64

20 besar:
Cluster_Label
Tinggi    20
Name: count, dtype: int64

50 besar:
Cluster_Label
Tinggi      49
Menengah     1
Name: count, dtype: int64

Terbaik per cluster:


,Ranking,Brand,Model,Price_IDR,CPU_Score,GPU_Score,RAM_GB,Storage_GB,Cluster_Label,Preference_Value
0,1,Asus,Rog Strix,29431000.0,14737,8782.0,16,1280.0,Tinggi,0.612532
37,38,Lenovo,Legion Y520-15IKBN,20501000.0,6887,6366.0,16,256.0,Menengah,0.543235
260,261,Lenovo,IdeaPad 320-15IKB,9481000.0,2719,1510.0,8,1024.0,Rendah,0.488413


### Cari laptop tertentu (opsional)
Ganti `keyword` dengan nama model yang ingin dilihat posisinya di ranking.

In [13]:
# Cari laptop tertentu di ranking (ganti kata kuncinya)
keyword = "Rog Strix"
ranking[ranking["Model"].str.contains(keyword, case=False, na=False)][show].head(10)

,Ranking,Brand,Model,Price_IDR,CPU_Score,GPU_Score,RAM_GB,Storage_GB,Cluster_Label,Preference_Value
0,1,Asus,Rog Strix,29431000.0,14737,8782.0,16,1280.0,Tinggi,0.612532
2,3,Asus,Rog Strix,24681000.0,14737,8782.0,8,1280.0,Tinggi,0.595064
3,4,Asus,Rog Strix,41781000.0,14737,8782.0,16,1280.0,Tinggi,0.578170
11,12,Asus,Rog Strix,32205000.0,12243,8782.0,8,1280.0,Tinggi,0.560152
59,60,Asus,ROG Strix,37031000.0,6887,10021.0,16,1280.0,Tinggi,0.535585
62,63,Asus,Rog Strix,46531000.0,6887,13510.0,16,1280.0,Tinggi,0.534687
78,79,Asus,Rog Strix,38948100.0,6887,10021.0,16,1280.0,Tinggi,0.529339
86,87,Asus,ROG Strix,31445000.0,6887,10021.0,8,1152.0,Tinggi,0.526000
153,154,Asus,ROG Strix,31331000.0,5079,10021.0,8,1152.0,Tinggi,0.510183
180,181,Asus,ROG Strix,33630000.0,5079,10021.0,8,1152.0,Tinggi,0.503297


## 11. Simpan hasil

In [14]:
ranking.drop(columns=["CPU_key", "GPU_key"]).to_excel("hasil_spk_rekomendasi_laptop.xlsx", index=False)
files.download("hasil_spk_rekomendasi_laptop.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>